### API Basics

1. idempotence: The response to a request is the same regardless of how many times the request is made. For example, if you make a GET request to retrieve data, you will get the same data back each time, as long as the underlying data has not changed.

Here's a short version that's easy to remember:

---

### Why Offset Pagination Becomes a Problem

With offset pagination, the database is told:

```sql
LIMIT 20 OFFSET 40
```

Meaning:

> Skip the first 40 rows and return the next 20.

The problem is that even though we only need 20 rows, the database still has to read the first 60 rows (40 skipped + 20 returned). As the offset grows (e.g., 100,000), the query becomes slower because more rows must be scanned and discarded.

It can also produce duplicate or missing results if new records are inserted between requests.

---

### How Cursor Pagination Solves It

Instead of saying:

> "Skip 40 rows"

we say:

> "Start after the last record I already saw."

Example:

First request:

```sql
SELECT *
FROM users
ORDER BY id DESC
LIMIT 3;
```

Returns:

```text
100, 99, 98
```

Response:

```json
{
  "users": [100, 99, 98],
  "next_cursor": 98
}
```

Second request:

```sql
SELECT *
FROM users
WHERE id < 98
ORDER BY id DESC
LIMIT 3;
```

Returns:

```text
97, 96, 95
```

---

### Why Cursor Pagination Is Better

* **Faster:** The database jumps directly to records after the cursor instead of scanning and skipping previous rows.
* **More reliable:** New records being inserted won't cause duplicates or missing results.
* **Scales well:** Performance remains consistent even with millions of rows.

### Easy Way to Remember

**Offset Pagination:**
*"Go to page 5."*
The database counts through pages 1–4 first.

**Cursor Pagination:**
*"I stopped at ID 98. Continue from there."*
The database resumes exactly where it left off.


In [5]:
%pip install requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [6]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/todos/2")

response.status_code

200

In [7]:
response.json()

{'userId': 1,
 'id': 2,
 'title': 'quis ut nam facilis et officia qui',
 'completed': False}

In [8]:
res =  requests.post("https://jsonplaceholder.typicode.com/todos", data={'title' : "Getting groceries", 'completed' : True, 'userid' : 12, id : 10})

In [9]:
res.json()

{'title': 'Getting groceries',
 'completed': 'True',
 'userid': '12',
 '<built-in function id>': '10',
 'id': 201}

In [10]:
response = requests.get("https://jsonplaceholder.typicode.com/todos?userid=12&id=10")

In [11]:
response.json()

[{'userId': 1,
  'id': 10,
  'title': 'illo est ratione doloremque quia maiores aut',
  'completed': True}]

## path variable
order/{id}/{size}

## request param
order?id=2&size=2

In [12]:
# response = requests.get("https://jsonplaceholder.typicode.com/todos/12")
# response.json()

userid = 12
response = requests.get("https://jsonplaceholder.typicode.com/todos/{userid}")
response.json()

{}

In [18]:
res = requests.delete("https://jsonplaceholder.typicode.com/todos", data = {'userid' : 2, 'id': 1})
print(res.status_code) /

404


In [19]:
res = requests.put("https://jsonplaceholder.typicode.com/todos/12", data = {'title' : "Getting groceries", 'completed' : True, 'userid' : 12, id : 1})
print(res.status_code)

200
